<div align="left" style="background-color: #008080; padding: 20px 10px;">
<h3><b>IDEAS - Institute of Data Engineering, Analytics and Science Foundation</b></h3>
<p>Summer Internship Program 2026</p>
<hr style="width:100%;">
<h3><b>Project Title:</b> Used Car Price Prediction Using Regression Models</h3>
<h4>Project Notebook</h4>

<blockquote style="border-left: 4px solid #4285F4; padding-left: 15px;">
  <strong>Created by:</strong> Suprava Das<br>
  <strong>Designation:</strong> Associate Software Developer
</blockquote>
<hr style="width:100%;">
</div>

### Question 1: Import Packages and Load the Dataset (1 Mark)

Import `pandas` as `pd`. Download the dataset `used_cars.csv` from https://drive.google.com/drive/folders/1gieHICVDBbUKMZiSF4YRQMAJic-w50JM?usp=sharing. Load the dataset `used_cars.csv` into a DataFrame named `df`.
Create a copy of the DataFrame named `df1` to preserve the original data for future reference.

**Expected Output:** The code cell should execute without any errors.

In [25]:
import pandas as pd

df = pd.read_csv("/content/used_cars.csv")
df1 = df.copy()

### Question 2: Shuffle and Display the Dataset (1 Mark)

Shuffle the dataset `df1` using `.sample(frac=1, random_state=42)` to ensure the data is randomly distributed. Display the top 10 rows of the shuffled dataset.

**Expected Output:** A table showing the first 10 rows of the randomized DataFrame.

In [26]:
df1.sample(frac=1, random_state=42)
print(df1.head(10))

      brand                            model  model_year       milage  \
0      Ford  Utility Police Interceptor Base        2013   51,000 mi.   
1   Hyundai                     Palisade SEL        2021   34,742 mi.   
2     Lexus                    RX 350 RX 350        2022   22,372 mi.   
3  INFINITI                 Q50 Hybrid Sport        2015   88,900 mi.   
4      Audi        Q3 45 S line Premium Plus        2021    9,835 mi.   
5     Acura                         ILX 2.4L        2016  136,397 mi.   
6      Audi             S3 2.0T Premium Plus        2017   84,000 mi.   
7       BMW                           740 iL        2001  242,000 mi.   
8     Lexus                   RC 350 F Sport        2021   23,436 mi.   
9     Tesla          Model X Long Range Plus        2020   34,000 mi.   

       fuel_type                                             engine  \
0  E85 Flex Fuel  300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...   
1       Gasoline                               3.8L V6

### Question 3: Dataset Overview and Missing Values (2 Marks)

Print the total number of data points (rows) in the dataset `df1`. Then, find and print the number of null (missing) values present in each column.

**Expected Output:** The total row count and a Series showing the count of nulls per column.

In [27]:
print("Total number of rows =", len(df1))

print("\nMissing values:")
print(df1.isnull().sum())

Total number of rows = 4009

Missing values:
brand             0
model             0
model_year        0
milage            0
fuel_type       170
engine            0
transmission      0
ext_col           0
int_col           0
accident        113
clean_title     596
price             0
dtype: int64


### Question 4: Standardize Missing or Invalid Strings (2 Marks)

For all columns with an `object` (string) data type in `df1`, perform the following replacements to standardize missing data:
- Replace the specific character `'–'` with the string `'Unknown'`.
- Replace the string `'nan'` with `'Unknown'`.
- Replace any entirely empty strings or strings consisting only of whitespace with `'Unknown'`.

**Expected Output:** The code cell should execute without errors, standardizing the text data.

In [14]:
obj_cols = df1.select_dtypes(include=['object']).columns
df1[obj_cols] = df1[obj_cols].replace(r'^\s*$', 'Unknown', regex=True)
df1[obj_cols] = df1[obj_cols].replace({'nan': 'Unknown', '–': 'Unknown'})

### Question 5: Feature Engineering - Car Age (1 Mark)

Create a new column named `car_age` in `df1` by calculating the age of each car. You can do this by subtracting the `model_year` from the current year (use `2026` as the current year).
After calculating the age, drop the original `model_year` column from the DataFrame.

**Expected Output:** The code cell should execute without errors.

In [15]:
if "model_year" in df1.columns:
    df1["car_age"] = 2026 - df1["model_year"]
    df1.drop("model_year", axis=1, inplace=True)



### Question 6: Format Numeric Columns (1 Mark)

Clean the following two columns so they can be used as numbers:
1. **milage:** Remove the string `' mi.'` and any commas (`,`), then convert the column to integers.
2. **price:** Remove currency symbols (`$`) and any commas (`,`), then convert the column to floats.

**Expected Output:** The code cell should execute without errors, changing both columns to numeric types.

In [16]:
df1["milage"] = df1["milage"].astype(str)
df1["milage"] = df1["milage"].str.replace(" mi.", "", regex=False)
df1["milage"] = df1["milage"].str.replace(",", "", regex=False)
df1["milage"] = df1["milage"].astype(int)
df1["price"] = df1["price"].astype(str)
df1["price"] = df1["price"].str.replace("$", "", regex=False)
df1["price"] = df1["price"].str.replace(",", "", regex=False)
df1["price"] = pd.to_numeric(df1["price"], errors="coerce")

### Question 7: Handle Low-Frequency Categories (2 Marks)

Some categorical columns have too many rare values. For both the `transmission` and `ext_col` columns, group categories that appear in **less than 1%** of the total dataset into a new category called `'others'`.
Additionally, drop the `clean_title` column entirely as it provides little variance.

**Expected Output:** The code cell should execute without errors.

In [17]:
threshold = len(df1) * 0.01

for col in ["transmission", "ext_col"]:

    counts = df1[col].value_counts()

    rare_values = counts[counts < threshold].index

    df1[col] = df1[col].replace(rare_values, "others")

df1.drop("clean_title", axis=1, inplace=True)

### Question 8: Remove Outliers and Apply Label Encoding (2 Marks)

Remove price outliers by filtering `df1` to keep only rows where the `price` is **below the 99th percentile**.
Next, import `LabelEncoder` from `sklearn.preprocessing` and apply it to convert all remaining categorical (`object`) columns into numerical values.

**Expected Output:** The code cell should execute without errors.

In [18]:
from sklearn.preprocessing import LabelEncoder

df1 = df1[df1['price'] < df1['price'].quantile(0.99)]
cat_cols = df1.select_dtypes(include=['object']).columns
le = LabelEncoder()
for col in cat_cols:
    df1[col] = le.fit_transform(df1[col])

### Question 9: Train-Test Split (1 Mark)

Define the `price` column as your target variable (`y`) and separate the remaining columns to form the feature set (`X`).
Split the features and target into training and testing datasets using an 80:20 ratio (`test_size=0.2`) and set `random_state=42`. Print the shapes of the resulting `X_train` and `X_test`.

**Expected Output:** A printout of the dimensions (shape) for the training and testing feature sets.

In [19]:
from sklearn.model_selection import train_test_split

df1 = pd.get_dummies(df1, dtype=int)
y = df1['price']
X = df1.drop(columns=['price'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (3174, 10)
X_test shape: (794, 10)


### Question 10: Model Training and Evaluation (4 Marks)

Import `RandomForestRegressor` from `sklearn.ensemble`, and `r2_score`, `mean_squared_error`, `mean_absolute_error` from `sklearn.metrics`.
Train the Random Forest model (with `n_estimators=100`, `random_state=42`) on the training data. Predict the prices for the test data and evaluate the model by printing the R² Score, MSE, and MAE.

**Expected Output:** Three numeric values representing the R² score, Mean Squared Error, and Mean Absolute Error of the model.

In [20]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
print(f"R² Score: {r2}")
print(f"Mean Squared Error (MSE): {mse}")
print(f"Mean Absolute Error (MAE): {mae}")


R² Score: 0.7286497669639422
Mean Squared Error (MSE): 319333182.33651596
Mean Absolute Error (MAE): 9462.83444584383


### Question 11: Load MNIST Data (3 Marks)

Load the MNIST dataset using `sklearn.datasets.load_digits`. Separate the dataset into features (`X`) and target labels (`y`).
Print the shape of the features and the target arrays.

**Expected Output:** The shape of `X` and `y`.

In [21]:
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.data
y = digits.target
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (1797, 64)
Target shape: (1797,)


### Question 12: K-Means Clustering (5 Marks)
Import `K Means` from `sklearn.cluster`. Initialize and fit the K-Means clustering model on the MNIST features (`X`).
Since we know there are 10 digits (0-9), set the number of clusters to 10. Assign the cluster labels to a variable `kmeans_labels`.

**Expected Output:** A successfully fitted K-Means model and the cluster labels array.

In [22]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=10, random_state=42)
kmeans.fit(X)
kmeans_labels = kmeans.labels_
print(kmeans_labels)

[5 7 7 ... 8 9 8]


### Question 13: F1 Score Evaluation for Clustering (5 Marks)

Evaluate the clustering performance using the F1 score. Since K-Means assigns arbitrary cluster labels, you will first need to map each cluster label to the most frequent true label in that cluster.
After mapping the labels, calculate and print the macro-averaged F1 score using `sklearn.metrics.f1_score`.

**Expected Output:** The calculated F1 score of the clustering.

In [23]:
import numpy as np
from sklearn.metrics import f1_score

mapped_labels = np.zeros_like(kmeans_labels)

for i in range(10):

    mask = (kmeans_labels == i)

    true_labels = y[mask]

    most_common = np.bincount(true_labels).argmax()

    mapped_labels[mask] = most_common

print("F1 Score =", f1_score(y, mapped_labels, average="macro"))

F1 Score = 0.8627854786352394
